Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [ ]:
import sys
for lib in libs.values():
  sys.path.append(lib)

Import libraries

In [ ]:
import os
import json
import copy
import pycomet
import numpy as np
import scipy as sp
import tensorflow as tf

from scipy.linalg import expm
from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from hypernet.thermophysics import specie_thermo as st_mod

In [ ]:
from pycomet.networks.chem.cg.matexp.blocks.operators import Operators

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x=None,
    y=None,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    n, d = y.shape if (x is not None) else yref.shape
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=n//20,
        ls='--',
        lw=1
    )
    for i in range(d):
        if (x is not None):
            ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
        if (xref is not None):
            ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
        ax.plot([], [], c=COLORS[i], label=names[i])
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [ ]:
with open("/home/zanardi/Codes/ML/PyCOMET/run/meno/physics/NRVE45/inputs.json") as file:
  phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

In [ ]:
T = np.array([1e4]).reshape(1,1)
kin._update_thermochem(T)

In [ ]:
rates = {}
for reaction in kin.reactions:
  r = str(reaction.param['process'])
  rates[r] = {k: getattr(reaction.fwd, k).numpy().squeeze() for k in ("A", "beta", "Ta")}

In [ ]:
tf.nest.map_structure(np.shape, rates)

Construct rate matrices

In [ ]:
kef, keb = 0.0, 0.0
for reaction in kin.reactions:
  if (reaction.param['process'] in (5,6)):
    kef = kef + reaction.fwd.k.numpy().squeeze()
    keb = keb + reaction.bwd.k.numpy().squeeze()
  elif (reaction.param['process'] == 2):
    kd = reaction.fwd.k.numpy().squeeze()
    kr = reaction.bwd.k.numpy().squeeze()
ke = kef + keb.T
ke = (ke.T - np.diag(np.sum(ke, axis=-1)))
ked = ke - np.diag(kd)

In [ ]:
qa, qm = [st.pf.tot.Q.numpy().reshape(-1) for st in sp_th.values()]
q = np.concatenate([qa, qm])

In [ ]:
ops_lay = Operators(
  rates=rates,
  use_corr=False
)

In [ ]:
qav = np.repeat(qa.reshape(1,-1), 100, axis=0)
qmv = np.repeat(qm.reshape(1,-1), 100, axis=0)
Tv = np.repeat(T.reshape(1,-1), 100, axis=0)
na = np.ones([1,1])
f = np.random.rand(100,45,3)*1e-3

In [ ]:
ked_pred, kr_pred = ops_lay((na, Tv, qav, qmv, f))

In [ ]:
print(ked_pred[0].numpy()[:4,:4])

In [ ]:
print(ked[:4,:4])

In [ ]:
np.mean(100 * np.abs(ked_pred[0] - ked) / np.abs(ked+1e-50))

Governing equations

In [ ]:
def construct_prob_mat(mapping):
  mapping = (mapping - np.amin(mapping)).astype(int)
  nb_lev = len(mapping)
  nb_comp = np.amax(mapping) + 1
  indices = (np.arange(nb_lev), mapping)
  P = np.zeros((nb_lev, nb_comp), dtype=int)
  P[indices] = 1
  return P

In [ ]:
mapping = np.array(tf.nest.flatten([[i for _ in range(9)] for i in range(5)]))
P = construct_prob_mat(mapping)

In [ ]:
Pq = qm.reshape(-1,1) * P
Qm = P.T @ qm
Pq /= Qm.reshape(1,-1)

In [ ]:
kefr = np.tril(Pq.T @ kef @ P, k=-1)
kebr = np.tril(P.T @ keb @ Pq, k=-1)
ker = kefr + kebr.T
ker = (ker.T - np.diag(np.sum(ker, axis=-1)))
kedr = ker - np.diag(Pq.T @ kd)
kedr

In [ ]:
Ked_pred = P.T @ ked @ Pq
Kr_pred = P.T @ kr
Ked_pred

In [ ]:
kedr - kc_pred[0]

In [ ]:
def _spmm_2d(a, b):
  b = tf.transpose(b)
  y = tf.sparse.sparse_dense_matmul(a, b)
  return tf.transpose(y)

def _spmm_3d(a, b):
  b = tf.transpose(b, perm=[1,2,0])
  ash, bsh = tf.shape(a), tf.shape(b)
  b = tf.reshape(b, (bsh[0], -1))
  y = tf.sparse.sparse_dense_matmul(a, b)
  y = tf.reshape(y, (ash[0], bsh[1], bsh[2]))
  return tf.transpose(y, perm=[2,0,1])

def compress_ops(k, w, q, P):
  # Compress linear operator
  # > Compress first dimension
  kc = _spmm_3d(P, k)
  # > Compress second dimension
  kc *= tf.expand_dims(q, axis=1)
  kc = tf.transpose(kc, perm=[0,2,1])
  kc = _spmm_3d(P, kc)
  kc = tf.transpose(kc, perm=[0,2,1])
  # > Normalize
  qc = _spmm_2d(P, q)
  kc *= (1/tf.expand_dims(qc, axis=1))
  # Compress linear source
  wc = _spmm_2d(P, w)
  return kc, wc

In [ ]:
def reduce(x, indices, nb_elem, axis=1):
  y = x
  if (indices is not None):
    nb_dim = len(tf.shape(x))
    axes = tf.range(0, nb_dim, delta=1)
    # Transpose in
    perm = tf.roll(axes, shift=-axis, axis=0)
    y = tf.transpose(y, perm=perm)
    # Reduce sum
    y = tf.math.unsorted_segment_sum(
      data=y,
      segment_ids=indices,
      num_segments=nb_elem
    )
    # Transpose out
    perm = tf.roll(axes, shift=axis, axis=0)
    y = tf.transpose(y, perm=perm)
  return y

In [ ]:
def compress_vars(k, w, qm):
  w, Qm = [reduce(z, indices=mapping, nb_elem=5, axis=1) for z in (w, qm)]
  # Compress linear operator
  # > Compress first dimension + Scaling
  k = reduce(k, indices=mapping, nb_elem=5, axis=1) * tf.expand_dims(qm, axis=1)
  # > Compress second dimension + Normalizing
  k = reduce(k, indices=mapping, nb_elem=5, axis=2) * tf.expand_dims(1/Qm, axis=1)
  return k, w

In [ ]:
# P = P.astype(np.float64)
# i = np.where(P.T>0)
# Psp = tf.sparse.SparseTensor(indices=np.vstack(i).T, values=P.T[i], dense_shape=P.T.shape)

In [ ]:
def construct_prob_mat(mapping):
  mapping = (mapping - np.amin(mapping)).astype(int)
  nb_comp_in, nb_comp_out = len(mapping), np.amax(mapping)+1
  # Construct probability matrix (sparse)
  indices = (mapping, np.arange(nb_comp_in))
  prob_mat = tf.sparse.SparseTensor(
    indices=np.vstack(indices).T,
    values=tf.ones(nb_comp_in, dtype="float64"),
    dense_shape=(nb_comp_out, nb_comp_in)
  )
  return prob_mat

mapping = np.array(tf.nest.flatten([[i for _ in range(9)] for i in range(5)]))
Psp = construct_prob_mat(mapping)

In [ ]:
s = 10000
ked_v = np.repeat(np.expand_dims(ked, 0), s, axis=0)
kr_v = np.repeat(np.expand_dims(kr, 0), s, axis=0)
qm_v = np.repeat(np.expand_dims(qm, 0), s, axis=0)

In [ ]:
kc_pred, krc_pred = compress_vars(ked_v, kr_v, qm_v)

In [ ]:
# kc_pred, krc_pred = compress_ops(ked_v, kr_v, qm_v, Psp)

In [ ]:
np.abs(krc_pred[0] - Kr_pred) / np.abs(Kr_pred)

In [ ]:
kc_pred[0]

In [ ]:
%timeit tf.matmul(PT, kr_v)

In [ ]:
%timeit tf.sparse.sparse_dense_matmul(PTsp, kr_v)

In [ ]:
ked_v = np.repeat(np.expand_dims(ked, -1), s, axis=-1)
ked_v = np.reshape(ked_v, (-1,ked_v.shape[1]*ked_v.shape[2]))
ked_v.shape

In [ ]:
tf.sparse.sparse_dense_matmul(PTsp, P)

In [ ]:
tf.random.uniform((45,1)) * P

In [ ]:
def fun(t, c):
    f = np.zeros_like(c)
    # Extract number densities
    n = c * const.UNA
    na, nm = n[0:1], n[1:]
    # Construct source terms
    we = np.matmul(ke,nm*na)
    wd = kd*nm*na - kr*na**3
    # Compose source terms
    f[0] = 2*np.sum(wd)
    f[1:] = we - wd
    return f / const.UNA

def solution(t, n0):
    res = sp.integrate.solve_ivp(
        fun=fun,
        t_span=[t[0],t[-1]],
        t_eval=t.squeeze(),
        y0=n0/const.UNA,
        method='LSODA',
        atol=1e-9,
        rtol=1e-7
    )
    return res.y.T * const.UNA

In [ ]:
ntest = solution(t, n[0])

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=ntest[1:],
    xref=t.squeeze()[1:],
    yref=n[1:],
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'log'],
    names=['O'] + [r'O$_2^{(%d)}$' % (i+1) for i in range(3)],
    figname=path_to_figs + "/solution"
)

Load master equation model

In [ ]:
model = FwdEulerME(
    processes=("exc", "dis"),
    fwd_rates={"kef": kef, "kd": kd},
    part_fn={"atom": qa, "molecule": qm}
)

In [ ]:
dnmdt_true = np.vstack([fun(t=0, c=ni/const.UNA) for ni in n])[:,1:]

In [ ]:
nt = len(t)
nb = n.shape[1]-1
x = {
    "t": np.ones((nt,1)),
    "n": n,
    "f": (np.ones((nt,nb,nb)), np.ones((nt,nb)))
}
xl = list(x.values())
dnmdt_pred = (model(xl) - n[:,1:]).numpy().squeeze() / const.UNA

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=dnmdt_pred[1:],
    xref=t.squeeze()[1:],
    yref=dnmdt_true[1:],
    labels=[r'$t$ [s]', r'$\omega$ [m$^{-3}$s$^{-1}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt_true.shape[1])],
    figname=path_to_figs + "/source"
)

In [ ]:
import tensorflow as tf

In [ ]:
n = 1000
m = 50
a = tf.random.uniform((n,m,m))
a = tf.cast(a, dtype=tf.float64)
b = tf.random.uniform((n,m))
b = tf.cast(b, dtype=tf.float64)

In [ ]:
%timeit tf.einsum("nij,njk->nik", a, b) + tf.einsum("nij,njk,nkl->nil", a, a, b)

In [ ]:
%timeit tf.matmul(a, b) + tf.matmul(tf.matmul(a, a), b)

In [ ]:
%timeit tf.matmul(a, b) + tf.matmul(a, tf.matmul(a, b))

In [ ]:
y1 = tf.matmul(b, a) + tf.matmul(tf.matmul(b, a), a)
y1[0,:,0]

In [ ]:
y2 = tf.matmul(a, b) + tf.matmul(tf.matmul(a, a), b)
y2[0,:,0]

In [ ]:
tf.reduce_mean(tf.abs(y1-y2))

In [ ]:
import sys
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet")

In [ ]:
import tensorflow as tf
from pycomet.operations.linalg.matrix_exp import expm

In [ ]:
n = 1000
m = 2
x = tf.random.uniform((n,m,m))
x = tf.cast(x, dtype=tf.float64)

In [ ]:
%timeit expm.expm(x, max_degree=1, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=2, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=4, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=8, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=12, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=1)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=2)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=3)

In [ ]:
%timeit expm.expm(x)

In [ ]:
import sys
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet")

In [ ]:
import numpy as np
import tensorflow as tf
from pycomet.operations.linalg.matrix_exp import expm, _compute_l1_norm

In [ ]:
n = 1000
m = 2

In [ ]:
norms = np.geomspace(1e-18, 1e3, 50)
for i, s in enumerate(norms):
  print("----------")
  x = tf.random.uniform((n,m,m))
  x = tf.cast(x, dtype=tf.float64)
  x = (s / _compute_l1_norm(x)) * x
  # ----------
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    y_true = tf.reduce_sum(tf.linalg.expm(x))
    y_pred = tf.reduce_sum(expm(x))
  dy_true_dx = tape.gradient(y_true, x)
  dy_pred_dx = tape.gradient(y_pred, x)
  # ----------
  err = 1e2 * tf.math.reduce_mean(tf.abs(y_true - y_pred) / y_true).numpy()
  derr = 1e2 * tf.math.reduce_mean(tf.abs(dy_true_dx - dy_pred_dx) / tf.abs(dy_true_dx)).numpy()
  print("i, s:", i, s)
  print("y_true:", tf.math.reduce_max(y_true))
  print("y_pred:", tf.math.reduce_max(y_pred))
  print("dy_true_dx:", tf.math.reduce_max(dy_true_dx))
  print("dy_pred_dx:", tf.math.reduce_max(dy_pred_dx))
  print("err:", err)
  print("derr:", derr)
  print("----------")

In [ ]:
x = tf.constant(tf.random.uniform((2,2)))

In [ ]:
for xi in x:
    print(xi[0]*x)

In [ ]:
nb_threads = 2
cpu = tf.config.list_physical_devices("CPU")[0]
tf.config.set_visible_devices(cpu)
tf.config.threading.set_inter_op_parallelism_threads(nb_threads)
tf.keras.backend.set_floatx('float64')
tf.config.threading.set_intra_op_parallelism_threads(nb_threads)

In [ ]:
def eigen(x):
  l, v = tf.linalg.eig(x)
  vinv = tf.linalg.inv(v)

In [ ]:
def compute_v1(x):
    l, v = tf.linalg.eig(x)
    vinv = tf.linalg.inv(v)
    expx = tf.einsum("...ij,...j,...jk->...ik", v, tf.math.exp(l), vinv)
    xinv = tf.einsum("...ij,...j,...jk->...ik", v, 1/l, vinv)
    return [tf.math.real(z) for z in (expx, xinv)]

def compute_v2(x):
    expx = tf.linalg.expm(x)
    xinv = tf.linalg.inv(x)
    return expx, xinv

In [ ]:
x = tf.random.uniform((10,10))
x = tf.cast(x, dtype="float64")

In [ ]:
res1 = compute_v1(x)
res2 = compute_v2(x)

In [ ]:
for i in range(len(res1)):
    print(tf.reduce_mean(tf.abs(res1[i] - res2[i])))

In [ ]:
n = 1000
m = 100

In [ ]:
x = tf.random.uniform((n,m,m))
x = tf.cast(x, dtype="float64")

In [ ]:
tf.linalg.inv(x) - tf.linalg.pinv(x)

In [ ]:
tf.linalg.pinv(x)

In [ ]:
%timeit tf.linalg.svd(x)

In [ ]:
%timeit tf.linalg.lu(x)

In [ ]:
%timeit tf.linalg.eig(x)

In [ ]:
%timeit tf.linalg.inv(x)

In [ ]:
%timeit tf.matmul(x, x)

In [ ]:
import scipy as sp
import numpy as np

In [ ]:
x = np.random.rand(5,5)
T, V = sp.linalg.schur(x)

In [ ]:
import scipy.linalg
import numpy.random
import numpy

# Setup
N = 100
H = 0.01*numpy.random.rand(N,N)
H = H+H.T # symmetric
S = numpy.random.rand(N,N)
S = S+S.T # symmetric
S = S.dot(S) # positive definite
Sinv = scipy.linalg.inv(S)
I = numpy.identity(N)

# Reference answer
ref = scipy.linalg.expm(Sinv.dot(H))

# Our solution
## first our cholesky factors
L = scipy.linalg.cholesky(S,lower=True)
Linv = scipy.linalg.solve_triangular(L, I, lower=True)

## now solve the normal eigenvalue problem
Hprime = Linv.dot(H).dot(Linv.T)
D, Xprime = scipy.linalg.eigh(Hprime)
D = numpy.diag(D)

## back subsite for X and X^{-1}
X = Linv.T.dot(Xprime)
Xinv = Xprime.T.dot(L.T)

our_answer = X.dot(scipy.linalg.expm(D)).dot(Xinv)

print(scipy.linalg.norm(ref - our_answer))

In [ ]:
A = Sinv.dot(H)
A.shape

In [ ]:
%timeit scipy.linalg.expm(A)

In [ ]:
import numpy as np

In [ ]:
x = np.tril(np.random.rand(5,5))
x

In [ ]:
np.where(x>0)

In [ ]:
x = np.random.rand(5,) - 0.5
x

In [ ]:
x[np.where(x>0)]

In [ ]:
x = tf.constant([1.8, 2.2], dtype="float32")
tf.cast(x, "int32")


In [ ]:
import sys
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet")

import json
import itertools
import numpy as np
import tensorflow as tf

from pycomet import utils

In [ ]:
class GroupsConnectivity(object):

  # Initialization
  # ===================================
  def __init__(
    self,
    groups_graph
  ):
    self.groups_graph = groups_graph
    if isinstance(self.groups_graph, str):
      with open(self.groups_graph) as file:
        self.groups_graph = json.load(file)
    # Last-to-current timescale connectivity
    self.last_to_current = self._convert_keys(
      self.connect_last_to_current(self.groups_graph)
    )
    # Current-to-last timescale connectivity
    self.current_to_last = self._convert_keys(
      self.connect_current_to_next(self.groups_graph)
    )

  def _convert_keys(self, d):
    return {str(k): v for (k, v) in d.items()}

  # Groups connectivity/indexing
  # ===================================
  def connect_current_to_next(
    self,
    graph,
    level=0,
    connectivity={}
  ):
    d = {}
    connectivity[level] = []
    if isinstance(graph, dict):
      for g in graph.values():
        if hasattr(g, "keys"):
          l = list(g.keys())
          l = tf.nest.map_structure(int, l)
          d = {**d, **g}
        else:
          l = d = g
        connectivity[level].append(l)
    else:
      connectivity[level] = None
    if (d != {}):
      return self.connect_current_to_next(
        d, level=level+1, connectivity=connectivity
      )
    else:
      return connectivity

  def connect_last_to_current(
    self,
    graph,
    ts={}
  ):
    nb_ts = utils.compute_dict_depth(graph)
    connectivity = self.graph_to_nested_list(graph)
    for l in range(-1,nb_ts):
      ts[l+1] = self.merge(connectivity, it_max=l+1)
    return ts

  def graph_to_nested_list(
    self,
    graph
  ):
    if isinstance(graph, dict):
      ls = []
      for g in graph.values():
        ls.append(self.graph_to_nested_list(g))
      return ls
    else:
      if isinstance(graph, (list, tuple, set)):
        return list(graph)
      else:
        utils.raise_err(
          ValueError,
          "The last level of the groups graph has to be a list."
        )

  def merge(
    self,
    ls,
    it=0,
    it_max=1
  ):
    if any(isinstance(l, list) for l in ls):
      if (it < it_max):
        ls = list(itertools.chain.from_iterable(ls))
        return self.merge(ls, it=it+1, it_max=it_max)
      else:
        return self.join(ls)

  def join(
    self,
    ls
  ):
    try:
      ls = [list(itertools.chain.from_iterable(l)) for l in ls]
      return self.join(ls)
    except:
      return ls

In [ ]:
class GroupsConnectivity(object):

  # Initialization
  # ===================================
  def __init__(
    self,
    groups_graph
  ):
    self.groups_graph = groups_graph
    if isinstance(self.groups_graph, str):
      with open(self.groups_graph) as file:
        self.groups_graph = json.load(file)
    # Current-to-next timescale connectivity index
    self.current_to_next = self.get_indices(
      self.connect_current_to_next(self.groups_graph)
    )
    # Last-to-current timescale connectivity index
    self.last_to_current = self.get_indices(
      self.connect_last_to_current(self.groups_graph)
    )

  # Groups graph indexing
  # ===================================
  def get_indices(self, connectivity):
    indices = {}
    for (l, lindices) in connectivity.items():
      if (lindices is not None):
        lindices = [np.full(len(idx), j) for (j,idx) in enumerate(lindices)]
        lindices = np.concatenate(lindices)
      indices[str(l)] = lindices
    return indices

  # Groups graph connectivity
  # ===================================
  def connect_current_to_next(
    self,
    graph,
    level=0,
    connectivity={}
  ):
    d = {}
    connectivity[level] = []
    if isinstance(graph, dict):
      for g in graph.values():
        if hasattr(g, "keys"):
          l = list(g.keys())
          l = tf.nest.map_structure(int, l)
          d = {**d, **g}
        else:
          l = d = g
        connectivity[level].append(l)
    else:
      connectivity[level] = None
    if (d != {}):
      return self.connect_current_to_next(
        d, level=level+1, connectivity=connectivity
      )
    else:
      return connectivity

  def connect_last_to_current(
    self,
    graph,
    ts={}
  ):
    nb_ts = utils.compute_dict_depth(graph)
    connectivity = self.graph_to_nested_list(graph)
    for l in range(-1,nb_ts):
      ts[l+1] = self.merge(connectivity, it_max=l+1)
    return ts

  def graph_to_nested_list(
    self,
    graph
  ):
    if isinstance(graph, dict):
      ls = []
      for g in graph.values():
        ls.append(self.graph_to_nested_list(g))
      return ls
    else:
      if isinstance(graph, (list, tuple, set)):
        return list(graph)
      else:
        utils.raise_err(
          ValueError,
          "The last level of the groups graph has to be a list."
        )

  def merge(
    self,
    ls,
    it=0,
    it_max=1
  ):
    if any(isinstance(l, list) for l in ls):
      if (it < it_max):
        ls = list(itertools.chain.from_iterable(ls))
        return self.merge(ls, it=it+1, it_max=it_max)
      else:
        return self.join(ls)

  def join(
    self,
    ls
  ):
    try:
      ls = [list(itertools.chain.from_iterable(l)) for l in ls]
      return self.join(ls)
    except:
      return ls

In [ ]:
filename = "/home/zanardi/Codes/ML/PyCOMET/run/meno/02.multi_pole/isothermal/datagen/data/groups_graph.json"

In [ ]:
conn = GroupsConnectivity(groups_graph=filename)

In [ ]:
conn.last_to_current

In [ ]:
conn.current_to_next